# Heaps: the always-ready "what's smallest?"

The general tour of lists, deques, dicts and sorting is in [`00-intro.ipynb`](00-intro.ipynb). This notebook is about one structure, from the inside out:

1. The problem heaps solve
2. The one rule, and how the tree lives in a plain list
3. Push: bubbling up
4. Pop: sinking down
5. Why `heapify` is O(n)
6. The `heapq` API
7. Max-heaps
8. Priorities with data: tuples and tie-breakers
9. Patterns: top-k, merging sorted lists, scheduling, Dijkstra
10. Top-k when the counts keep changing
11. Costs at a glance

## 1. The problem

Items keep arriving and leaving, and at any moment you want the **smallest one**: the most urgent task, the nearest node, the earliest deadline.

| Approach | Get smallest | Insert | Remove smallest |
|---|---|---|---|
| Unsorted list | O(n) scan | O(1) | O(n) |
| Sorted list | O(1) | O(n) (shifting) | O(1) from the end, if kept in reverse order |
| **Heap** | **O(1)** | **O(log n)** | **O(log n)** |

A heap is the compromise: **not fully sorted, just sorted enough.**

## 2. The shape: one rule, stored in a plain list

A **min-heap** is a binary tree with exactly one rule:

> **Every parent is smaller than (or equal to) its children.**

Siblings and cousins can be in any order. The only guarantee runs *vertically*, from the root down each path. It follows that the smallest item is always at the top.

```
                1
             /     \
            3       2
          /   \    /  \
         7     4  5    9
        / \
      10   8
```

The tree is always **complete** (filled level by level, left to right, no gaps), so it packs into an ordinary list, level by level. The family relationships become arithmetic:

- children of index `i`: `2*i + 1` and `2*i + 2`
- parent of index `i`: `(i - 1) // 2`

That's why `heapq` has no `Heap` class: **a heap is just a list you only touch through `heapq` functions.**

Two small helpers, used throughout the notebook:

In [1]:
def show_tree(h):
    # Print a list-based heap level by level.
    i, width, level = 0, 1, 0
    while i < len(h):
        print(f"level {level}: {h[i:i + width]}")
        i, width, level = i + width, width * 2, level + 1

def is_heap(h):
    # Check the rule: every parent <= each of its children.
    return all(h[(i - 1) // 2] <= h[i] for i in range(1, len(h)))

h = [1, 3, 2, 7, 4, 5, 9, 10, 8]
show_tree(h)
print("is_heap:", is_heap(h))

level 0: [1]
level 1: [3, 2]
level 2: [7, 4, 5, 9]
level 3: [10, 8]
is_heap: True


In [2]:
i = 1
print(f"node at index {i} = {h[i]}")
print(f"children at {2*i + 1} and {2*i + 2}: {h[2*i + 1]}, {h[2*i + 2]}")
print(f"parent at {(i - 1) // 2}: {h[(i - 1) // 2]}")

node at index 1 = 3
children at 3 and 4: 7, 4
parent at 0: 1


### Why the height matters

A complete tree with n items has about **log₂ n levels**. Every heap operation walks *one path* from top to bottom or bottom to top, never across. So each operation takes about as many steps as there are levels:

In [3]:
import math

for n in (10, 1_000, 1_000_000, 1_000_000_000):
    print(f"n = {n:>13,}  ->  {math.floor(math.log2(n)) + 1:>2} levels")

n =            10  ->   4 levels
n =         1,000  ->  10 levels
n =     1,000,000  ->  20 levels
n = 1,000,000,000  ->  30 levels


## 3. Push: bubble up (sift-up)

To add a value:

1. Put it at the **end** of the list. The tree stays complete.
2. Compare it with its parent. If it's smaller, **swap**.
3. Repeat until the parent is smaller, or it reaches the top.

At most one swap per level: **O(log n)**. Here is a hand-written version that prints each step. `heapq.heappush` does the same thing, in C.

In [4]:
def push_traced(h, x):
    h.append(x)
    i = len(h) - 1
    print(f"append {x} at index {i}: {h}")
    while i > 0:
        p = (i - 1) // 2
        if h[i] < h[p]:
            h[i], h[p] = h[p], h[i]
            print(f"  {x} < parent {h[i]} -> swap indexes {i} and {p}: {h}")
            i = p
        else:
            print(f"  parent {h[p]} <= {x} -> stop")
            break

h = [1, 3, 2, 7, 4, 5, 9, 10, 8]
push_traced(h, 0)
show_tree(h)
print("is_heap:", is_heap(h))

append 0 at index 9: [1, 3, 2, 7, 4, 5, 9, 10, 8, 0]
  0 < parent 4 -> swap indexes 9 and 4: [1, 3, 2, 7, 0, 5, 9, 10, 8, 4]
  0 < parent 3 -> swap indexes 4 and 1: [1, 0, 2, 7, 3, 5, 9, 10, 8, 4]
  0 < parent 1 -> swap indexes 1 and 0: [0, 1, 2, 7, 3, 5, 9, 10, 8, 4]
level 0: [0]
level 1: [1, 2]
level 2: [7, 3, 5, 9]
level 3: [10, 8, 4]
is_heap: True


In [5]:
h = [1, 3, 2, 7, 4, 5, 9, 10, 8]
push_traced(h, 6)    # a bigger value stops early

append 6 at index 9: [1, 3, 2, 7, 4, 5, 9, 10, 8, 6]
  parent 4 <= 6 -> stop


## 4. Pop: sink down (sift-down)

To remove the smallest (the root):

1. Take the root: that's the answer.
2. Move the **last** item of the list into the root position. The tree stays complete, but the rule is probably broken.
3. Compare it with its two children and **swap with the smaller child**, so the new parent is smaller than both.
4. Repeat until it's smaller than both children, or it has none.

Again one path down: **O(log n)**.

In [6]:
def pop_traced(h):
    top = h[0]
    last = h.pop()
    if not h:
        return top
    h[0] = last
    print(f"took {top}; moved last item {last} to the root: {h}")
    i, n = 0, len(h)
    while True:
        left, right, smallest = 2 * i + 1, 2 * i + 2, i
        if left < n and h[left] < h[smallest]:
            smallest = left
        if right < n and h[right] < h[smallest]:
            smallest = right
        if smallest == i:
            print(f"  {h[i]} is <= its children -> stop")
            break
        h[i], h[smallest] = h[smallest], h[i]
        print(f"  swap with smaller child {h[i]} (index {smallest}): {h}")
        i = smallest
    return top

h = [1, 3, 2, 7, 4, 5, 9, 10, 8]
print("returned:", pop_traced(h))
show_tree(h)
print("is_heap:", is_heap(h))

took 1; moved last item 8 to the root: [8, 3, 2, 7, 4, 5, 9, 10]
  swap with smaller child 2 (index 2): [2, 3, 8, 7, 4, 5, 9, 10]
  swap with smaller child 5 (index 5): [2, 3, 5, 7, 4, 8, 9, 10]
  8 is <= its children -> stop
returned: 1
level 0: [2]
level 1: [3, 5]
level 2: [7, 4, 8, 9]
level 3: [10]
is_heap: True


The hand-written versions behave like `heapq`. Popping everything returns the items in sorted order:

In [7]:
import heapq
import random

data = random.sample(range(100), 12)
print("input:          ", data)

ours = []
for x in data:
    ours.append(x)
    i = len(ours) - 1                      # silent sift-up
    while i > 0 and ours[i] < ours[(i - 1) // 2]:
        p = (i - 1) // 2
        ours[i], ours[p] = ours[p], ours[i]
        i = p

theirs = []
for x in data:
    heapq.heappush(theirs, x)

print("same heap list: ", ours == theirs)
print("pop all:        ", [heapq.heappop(theirs) for _ in range(len(data))])

input:           [46, 36, 0, 42, 62, 32, 69, 43, 7, 72, 55, 4]
same heap list:  True
pop all:         [0, 4, 7, 32, 36, 42, 43, 46, 55, 62, 69, 72]


## 5. Why `heapify` is O(n), not O(n log n)

Pushing n items one by one costs O(n log n). `heapq.heapify` is cleverer: it **sinks** nodes, starting from the last parent and working back up to the root.

- Half the nodes are leaves: they can't sink, so they cost nothing.
- A quarter sit just above the leaves: they sink at most 1 level.
- An eighth sink at most 2 levels, and so on.

The total is n/4 × 1 + n/8 × 2 + n/16 × 3 + … ≈ **n**. Most nodes live near the bottom, where sinking is short.

Let's count swaps on the worst-case input for both methods: a descending list (every new item is the smallest so far).

In [8]:
def swaps_push_one_by_one(data):
    h, swaps = [], 0
    for x in data:
        h.append(x)
        i = len(h) - 1
        while i > 0 and h[i] < h[(i - 1) // 2]:
            p = (i - 1) // 2
            h[i], h[p] = h[p], h[i]
            i, swaps = p, swaps + 1
    return swaps

def swaps_heapify(data):
    h, swaps = list(data), 0
    n = len(h)
    for start in range(n // 2 - 1, -1, -1):      # last parent back to the root
        i = start
        while True:
            left, right, smallest = 2 * i + 1, 2 * i + 2, i
            if left < n and h[left] < h[smallest]:
                smallest = left
            if right < n and h[right] < h[smallest]:
                smallest = right
            if smallest == i:
                break
            h[i], h[smallest] = h[smallest], h[i]
            i, swaps = smallest, swaps + 1
    return swaps

n = 2**16 - 1
worst = list(range(n, 0, -1))
print(f"n = {n:,}")
print(f"push one by one: {swaps_push_one_by_one(worst):>9,} swaps  (~ n log n)")
print(f"heapify        : {swaps_heapify(worst):>9,} swaps  (~ n)")

n = 65,535
push one by one:   917,506 swaps  (~ n log n)
heapify        :    65,519 swaps  (~ n)


## 6. The `heapq` API

| Function | What it does | Cost |
|---|---|---|
| `heappush(h, x)` | Add `x` | O(log n) |
| `heappop(h)` | Remove and return the smallest | O(log n) |
| `h[0]` | Peek at the smallest | O(1) |
| `heapify(lst)` | Turn a list into a heap, in place | O(n) |
| `heappushpop(h, x)` | Push `x`, then pop the smallest | O(log n) |
| `heapreplace(h, x)` | Pop the smallest, then push `x` | O(log n) |
| `nlargest(k, it, key=)` / `nsmallest(k, it, key=)` | Top k without sorting everything | O(n log k) |
| `merge(*sorted_iterables)` | Lazily merge sorted inputs | O(N log k) |

In [9]:
import heapq

h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
print("heap list:", h)          # not sorted, just heap-ordered
print("peek h[0]:", h[0])
print("heappop  :", heapq.heappop(h), "->", h)

nums = [9, 4, 7, 1, 6]
heapq.heapify(nums)             # in place, returns None
print("heapified:", nums)

heap list: [1, 3, 8, 5]
peek h[0]: 1
heappop  : 1 -> [3, 5, 8]
heapified: [1, 4, 7, 9, 6]


`heappushpop` and `heapreplace` look alike but differ in order, which matters when the new item is smaller than the current minimum:

In [10]:
import heapq

a = [2, 5, 9]
print("heappushpop(a, 1) ->", heapq.heappushpop(a, 1), a)   # 1 goes in, then the smallest (1) comes out

b = [2, 5, 9]
print("heapreplace(b, 1) ->", heapq.heapreplace(b, 1), b)   # 2 comes out first, then 1 goes in

heappushpop(a, 1) -> 1 [2, 5, 9]
heapreplace(b, 1) -> 2 [1, 5, 9]


In [11]:
import heapq

prices = {"apple": 1.2, "kiwi": 0.4, "mango": 2.5, "pear": 0.9, "fig": 3.1}
print("3 cheapest:     ", heapq.nsmallest(3, prices.items(), key=lambda kv: kv[1]))
print("2 most expensive:", heapq.nlargest(2, prices.items(), key=lambda kv: kv[1]))

3 cheapest:      [('kiwi', 0.4), ('pear', 0.9), ('apple', 1.2)]
2 most expensive: [('fig', 3.1), ('mango', 2.5)]


## 7. Max-heaps

`heapq` is a **min-heap**. The portable way to get a max-heap is to store **negated** values: push `-x`, and negate again when you pop.

(Python 3.14 also added public max-heap functions such as `heapq.heappush_max` and `heapq.heappop_max`. Negation works on every version.)

In [12]:
import heapq

nums = [5, 1, 8, 3]
max_heap = [-x for x in nums]
heapq.heapify(max_heap)

print("largest:", -max_heap[0])
print("pop order:", [-heapq.heappop(max_heap) for _ in range(len(nums))])

largest: 8
pop order: [8, 5, 3, 1]


## 8. Priorities with data: tuples and tie-breakers

Push tuples. Tuples compare element by element, so the **first field is the priority**, and later fields only matter on ties.

In [13]:
import heapq

tasks = []
heapq.heappush(tasks, (2, "write report"))
heapq.heappush(tasks, (1, "fix outage"))
heapq.heappush(tasks, (2, "answer email"))   # same priority: the string breaks the tie

while tasks:
    print(heapq.heappop(tasks))

(1, 'fix outage')
(2, 'answer email')
(2, 'write report')


The catch: if two priorities are equal and the next field **can't be compared** (a dict, a custom object), you get an error:

In [14]:
import heapq

h = []
heapq.heappush(h, (1, {"name": "fix outage"}))
try:
    heapq.heappush(h, (1, {"name": "page on-call"}))
except TypeError as e:
    print("TypeError:", e)

TypeError: '<' not supported between instances of 'dict' and 'dict'


The fix: put a **unique counter** between the priority and the payload. Ties are then broken by the counter, which also keeps first-in, first-out order among equals.

In [15]:
import heapq
import itertools

counter = itertools.count()
h = []
for priority, task in [(1, {"name": "fix outage"}), (1, {"name": "page on-call"}), (0, {"name": "security alert"})]:
    heapq.heappush(h, (priority, next(counter), task))

while h:
    priority, _, task = heapq.heappop(h)
    print(priority, task["name"])

0 security alert
1 fix outage
1 page on-call


## 9. The classic heap patterns

When a problem says *"the k smallest/largest"*, *"the next earliest"*, *"repeatedly take the best"* or *"merge sorted things"*, think heap.

### 9.1 Top-k from a stream, without sorting everything - handwritten version of heapq.nlargest

"The k largest of n items." Sorting costs O(n log n). Instead:

* Pick the first k elements of the array, and create a min-size heap, h, of lenght K. 
* The smller number is always at the h[0]
* Next, try adding new numbers to the heap: only if they are larger than the smaller, h[0].
* when large numbers arrive, the smallest is always removed; so at any stage, we are left with the k largest numbers seen so far.
* **Cost: O(n log k) time, O(k) memory.** Great when k is small or the data doesn't fit in memory.

In [ ]:
import heapq

def top_k_traced(stream, k):
    h = []
    for x in stream:
        if len(h) < k:
            heapq.heappush(h, x)
            print(f"{x:>3}: heap not full, push       -> {sorted(h)}")
        elif x > h[0]:
            # Push out the smallest number if x is bigger... only the k larger values will survive!
            out = heapq.heapreplace(h, x)   # drop the guard, admit x
            print(f"{x:>3}: beats guard {out:>3}, swap in  -> {sorted(h)}")
        else:
            print(f"{x:>3}: <= guard {h[0]:>3}, ignore     -> {sorted(h)}")
    return sorted(h, reverse=True)

stream = [5, 12, 3, 40, 7, 25, 1, 30]
print("top 3:", top_k_traced(stream, 3))
print("check:", heapq.nlargest(3, stream))

  5: heap not full, push       -> [5]
 12: heap not full, push       -> [5, 12]
  3: heap not full, push       -> [3, 5, 12]
 40: beats guard   3, swap in  -> [5, 12, 40]
  7: beats guard   5, swap in  -> [7, 12, 40]
 25: beats guard   7, swap in  -> [12, 25, 40]
  1: <= guard  12, ignore     -> [12, 25, 40]
 30: beats guard  12, swap in  -> [25, 30, 40]
top 3: [40, 30, 25]
check: [40, 30, 25]


### 9.2 Merging k sorted lists

Put the first item of each list in a heap. Pop the smallest, then push the next item **from the same list**. Each item passes through a heap of size k: **O(N log k)** for N items in total.

In [ ]:
import heapq

def merge_sorted(lists):
    h = [
        (lst[0], i, 0) # (value, which list, position / or values popped from that list)
        for i, lst in enumerate(lists) if lst]   
    heapq.heapify(h)
    out = []
    while h:
        value, i, j = heapq.heappop(h)
        out.append(value)
        # If the list from which we took the smallest value still has values, pick a new one.
        if j + 1 < len(lists[i]):
            heapq.heappush(h, (lists[i][j + 1], i, j + 1))
    return out

lists = [[1, 4, 9], [2, 3, 10, 11], [5, 6]]
print(merge_sorted(lists))
print(list(heapq.merge(*lists)))   # the built-in, lazy version

[1, 2, 3, 4, 5, 6, 9, 10, 11]
[1, 2, 3, 4, 5, 6, 9, 10, 11]


### 9.3 Scheduling: how many meeting rooms?

Sort the meetings by start time. Keep a min-heap of the **end times** of meetings in progress: the top is the room that frees up first. If the next meeting starts after that, reuse the room (pop); either way, push the new end time. The heap's largest size is the answer.

Cost: O(n log n), dominated by the sort.

In [18]:
import heapq

def min_rooms(meetings):
    ends = []
    peak = 0
    for start, end in sorted(meetings):
        if ends and ends[0] <= start:
            freed = heapq.heappop(ends)
            note = f"reuse room freed at {freed}"
        else:
            note = "open a new room"
        heapq.heappush(ends, end)
        peak = max(peak, len(ends))
        print(f"meeting {start:>2}-{end:<2}: {note:<24} rooms in use: {len(ends)}")
    return peak

meetings = [(9, 10), (9, 12), (10, 11), (11, 13), (12, 14)]
print("rooms needed:", min_rooms(meetings))

meeting  9-10: open a new room          rooms in use: 1
meeting  9-12: open a new room          rooms in use: 2
meeting 10-11: reuse room freed at 10   rooms in use: 2
meeting 11-13: reuse room freed at 11   rooms in use: 2
meeting 12-14: reuse room freed at 12   rooms in use: 2
rooms needed: 2


### 9.4 Shortest paths: Dijkstra, with lazy deletion

A heap of `(distance, node)` always hands you the nearest unexplored node.

A heap can't efficiently update an item in its middle (finding it is O(n)). So when a shorter route to a node turns up, push a **new** entry and leave the old one in place. When the old one surfaces later, it's **stale**: skip it. This is called **lazy deletion**.

In [19]:
import heapq

graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("D", 1)],
    "C": [("B", 2), ("D", 5)],
    "D": [],
}

def dijkstra(graph, start):
    dist = {start: 0}
    h = [(0, start)]
    stale = 0
    while h:
        d, u = heapq.heappop(h)
        if d > dist.get(u, float("inf")):
            stale += 1
            print(f"  skip stale entry ({d}, {u!r})")
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(h, (nd, v))
    return dist, stale

dist, stale = dijkstra(graph, "A")
print("distances:", dist)
print("stale entries skipped:", stale)

  skip stale entry (4, 'B')
  skip stale entry (6, 'D')
distances: {'A': 0, 'B': 3, 'C': 1, 'D': 4}
stale entries skipped: 2


## 10. Top-k when the counts keep changing

The problem: maintain `{post_id: count}` as events arrive, and on demand return the `k` posts with the highest counts (ties broken by `post_id`, ascending).

Heaps handle the pairs fine: store `(-count, post_id)` tuples, so the heap orders by count and `post_id` only breaks ties. The real difficulty is that **counts change**, and a heap can't update an entry in its middle. That's also why the size-k heap from §9.1 doesn't apply directly: it assumes each value arrives once and never changes.

So the choice depends on how often you update compared with how often you query. Three designs, all behind the same two methods (`record`, `top_k`) so the internals can be swapped without touching callers.

### Option 1: dict, compute the top k when asked (the default)

O(1) per update, O(n log k) per query. Best when updates are frequent and queries rare. `nsmallest` with key `(-count, post_id)` gives "highest count first, smallest id on ties" (you can't negate a string id, so `nlargest` can't express that tie-break).

In [ ]:
import heapq
from collections import Counter

class RankingDict:
    def __init__(self):
        self.counts = Counter()

    def record(self, post_id):
        self.counts[post_id] += 1                       # O(1)

    def top_k(self, k):
        # self.counts.most_common(k) # O(n log k) # But you can bnot control the behaviour at ties.
        best = heapq.nsmallest(k, self.counts.items(),  # O(n log k)
                               key=lambda kv: (-kv[1], kv[0]))
        return [(pid, c) for pid, c in best]

events = ["p2", "p1", "p3", "p1", "p2", "p1", "p4", "p3", "p2"]
r1 = RankingDict()
for e in events:
    r1.record(e)
print("counts:", dict(r1.counts))
print("top 2 :", r1.top_k(2))
print("top 10:", r1.top_k(10))     # k larger than the number of posts is fine

counts: {'p2': 3, 'p1': 3, 'p3': 2, 'p4': 1}
top 2 : [('p1', 3), ('p2', 3)]
top 10: [('p1', 3), ('p2', 3), ('p3', 2), ('p4', 1)]


`Counter.most_common(k)` is the one-liner version, but on ties it keeps first-seen order rather than ordering by id:

In [21]:
print(r1.counts.most_common(3))

[('p2', 3), ('p1', 3), ('p3', 2)]


### Option 2: dict + heap with lazy deletion (both operations frequent)

On every update, push a fresh `(-count, post_id)` entry and leave the old one in the heap as **stale**. On a query, pop entries, skip any whose count no longer matches the dict, and push the valid ones back.

- Update: O(log m), where m is the heap size.
- Query: about O(k log m), plus the stale entries it removes. Each stale entry is popped at most once, ever.
- The heap grows by one entry per update, so rebuild it from the dict when it gets too big (O(n) with `heapify`).
- `seen` matters only if counts can also go down, so that an old count value could reappear.

In [ ]:
import heapq

class RankingLazyHeap:
    def __init__(self):
        self.counts = {}
        self.heap = []                      # (-count, post_id); may hold stale entries

    def record(self, post_id):
        # get new count - and store in dict.
        if post_id not in self.counts:
            self.counts[post_id] = 0
        self.counts[post_id] += 1
        c = self.counts[post_id]

        # update a heap (with potential duplicates for a fiven post_id)!
        heapq.heappush(self.heap, (-c, post_id))            # O(log m)

        # Once the heap is too large, reset it.
        if len(self.heap) > 2 * len(self.counts) + 16:      # too many stale entries
            self.heap = [(-c, pid) for pid, c in self.counts.items()]
            heapq.heapify(self.heap)                         # O(n)

    def top_k(self, k):
        # All we need to do is to pop the top items form the heap... with one fix: as the heap can contain multiple
        # items per post-id (the stale), we pop only if the post has not been seen yet.
        result, keep, seen = [], [], set()
        while self.heap and len(result) < k:
            entry = heapq.heappop(self.heap)
            neg_c, pid = entry
            if -neg_c != self.counts[pid] or pid in seen:
                continue                    # stale or duplicate: dropped for good
            seen.add(pid)
            result.append((pid, -neg_c))
            keep.append(entry)
        for entry in keep:                  # valid entries go back in
            heapq.heappush(self.heap, entry)
        return result

r2 = RankingLazyHeap()
for e in events:
    r2.record(e)
print("heap before query:", sorted(r2.heap))
print("top 4:", r2.top_k(4))
print("heap after query: ", sorted(r2.heap))

heap before query: [(-3, 'p1'), (-3, 'p2'), (-2, 'p1'), (-2, 'p2'), (-2, 'p3'), (-1, 'p1'), (-1, 'p2'), (-1, 'p3'), (-1, 'p4')]
top 4: [('p1', 3), ('p2', 3), ('p3', 2), ('p4', 1)]
heap after query:  [(-3, 'p1'), (-3, 'p2'), (-2, 'p3'), (-1, 'p4')]


Before the query, the heap held one entry per update, so several per post. To find the top 4, the query had to pop past the stale ones, and it dropped them for good. Only the four current entries were pushed back.

### Option 3: dict + sorted list with `bisect` (queries very frequent)

A dict has no order, so `bisect` can't search it directly. "Dict plus bisect" really means keeping a second, **sorted list** of `(-count, post_id)`. Queries are almost free, but each update removes and re-inserts one tuple, which **shifts** elements: O(n) per update.

In [23]:
import bisect

class RankingSortedList:
    def __init__(self):
        self.counts = {}
        self.ranked = []                    # sorted list of (-count, post_id)

    def record(self, post_id):
        old = self.counts.get(post_id, 0)
        if old:
            i = bisect.bisect_left(self.ranked, (-old, post_id))   # O(log n) to find
            del self.ranked[i]                                      # O(n) shift
        self.counts[post_id] = old + 1
        bisect.insort(self.ranked, (-(old + 1), post_id))           # O(n) shift

    def top_k(self, k):
        return [(pid, -neg_c) for neg_c, pid in self.ranked[:k]]   # O(k)

r3 = RankingSortedList()
for e in events:
    r3.record(e)
print("ranked list:", r3.ranked)
print("top 2:", r3.top_k(2))

ranked list: [(-3, 'p1'), (-3, 'p2'), (-2, 'p3'), (-1, 'p4')]
top 2: [('p1', 3), ('p2', 3)]


### All three agree

Feed the same random stream to all three and compare answers after every few events:

In [24]:
import random

random.seed(7)
impls = [RankingDict(), RankingLazyHeap(), RankingSortedList()]
mismatches = 0
for step in range(20_000):
    pid = f"p{random.randint(1, 300)}"
    for impl in impls:
        impl.record(pid)
    if step % 500 == 0:
        k = random.randint(1, 10)
        answers = [impl.top_k(k) for impl in impls]
        mismatches += answers[0] != answers[1] or answers[0] != answers[2]

print("mismatches:", mismatches)
print("top 5:", impls[0].top_k(5))
print(f"posts: {len(impls[1].counts)}, lazy heap size: {len(impls[1].heap)} (kept bounded by rebuilds)")

mismatches: 0
top 5: [('p165', 95), ('p115', 92), ('p47', 91), ('p114', 90), ('p156', 86)]
posts: 300, lazy heap size: 356 (kept bounded by rebuilds)


| Approach | Update | Top-k query | Best when |
|---|---|---|---|
| Dict, compute at query time | O(1) | O(n log k) | Default; updates frequent, queries rare |
| Dict + lazy heap | O(log m) | ~O(k log m) | Both frequent; many posts, small k |
| Dict + sorted list (`bisect`) | O(n) | O(k) | Queries constant, updates rare |

The shifting in Option 3 is a fast C memory move, so in practice it's fine for tens of thousands of posts, but it's still the worst of the three for updates as n grows. (The third-party `sortedcontainers.SortedList` makes those updates O(log n), if it's available.)

A fourth design exists for counts that only ever go up by one: a dict from count to the set of ids with that count (the "LFU cache" structure), with O(1) updates. It's more code to get right.

## 11. Heap costs at a glance

| Operation | Cost |
|---|---|
| Peek at the smallest (`h[0]`) | O(1) |
| `heappush`, `heappop`, `heapreplace`, `heappushpop` | O(log n) |
| `heapify` | O(n) |
| Find or remove an arbitrary item | O(n): avoid it, use lazy deletion |
| `nlargest(k)` / `nsmallest(k)` | O(n log k) |
| `merge` of k sorted inputs, N items | O(N log k) |

**The picture to keep:** a heap keeps just enough order that the smallest is always on top, and every change walks only one short path, about log₂ n steps, between the root and a leaf.